In [1]:
import torch
import torch.nn as nn

class MoSE_Module(nn.Module):
    def __init__(self, embed_dim, num_experts=1024, top_k=512, h=96, w=96):
        super(MoSE_Module, self).__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        # 1. The Shape Dictionary: 'n' learnable shape experts.
        # Shape: (num_experts, h, w)
        self.shape_experts = nn.Parameter(torch.randn(num_experts, h, w))
        
        # 2. The Gating Network: A lightweight CNN to generate pixel-wise weights.
        self.gating_network = nn.Sequential(
            nn.Conv2d(in_channels=embed_dim, out_channels=num_experts, kernel_size=1)
        )
        
    def forward(self, e_i):
        # e_i is the SAM image embedding: (Batch, Channels, Height, Width)
        
        # Step 1: Generate routing weights for every expert at every pixel
        G = self.gating_network(e_i) 
        
        # Step 2: Sparse Activation (Top-K Selection based on absolute values)
        # Find the indices of the top-k highest responses
        topk_values, topk_indices = torch.topk(torch.abs(G), self.top_k, dim=1)
        
        # Create an empty tensor of zeros, then fill only the top-k selected weights back in
        sparse_G = torch.zeros_like(G)
        sparse_G.scatter_(dim=1, index=topk_indices, src=G.gather(dim=1, index=topk_indices))
        
        # Step 3: Generate the final Shape Map
        # Multiply the sparse weights by the shape experts and sum them up
        shape_map = torch.sum(sparse_G * self.shape_experts.unsqueeze(0), dim=1)
        
        # Step 4: Normalize to a [0, 1] prompt format for SAM
        shape_prompt = torch.sigmoid(shape_map)
        
        return shape_prompt, sparse_G

In [2]:
# 1. Initialize our model (this builds the blueprint we just saved)
model = MoSE_Module(embed_dim=256)

# 2. Create a fake "dummy" image tensor (Batch Size: 1, Channels: 256, Height: 96, Width: 96)
dummy_image_features = torch.randn(1, 256, 96, 96)

# 3. Pass the dummy image through our model's forward pass
shape_prompt, sparse_weights = model(dummy_image_features)

# 4. Print the dimensions of the outputs to prove it worked!
print("Success! The model processed the data.")
print("Output Shape Map size:", shape_prompt.shape)
print("Sparse Weights size:", sparse_weights.shape)

Success! The model processed the data.
Output Shape Map size: torch.Size([1, 96, 96])
Sparse Weights size: torch.Size([1, 1024, 96, 96])
